# 02 · MAE 训练、模型结构与遮挡重建评估

本 Notebook 只评价 **MAE 训练任务的出口：被遮挡区域的像素预测**。不使用基因、药物或 embedding 检索标签。请在服务器的 `Python (cellvit)` 内核中运行，代码与训练版本保持一致。

流程：配置与日志 → 训练曲线 → 加载 checkpoint → 模型结构 → 官方重建评估 → 各通道结果 → 重建图 → 可选灰度诊断。

- 输入：六通道图像 `[B,6,512,512]`，像素为 `[0,1]`。
- Encoder 可见比例由 checkpoint 中的 `mask_ratio` 决定；Decoder 预测全部 patch，误差只统计隐藏位置。
- 默认目标是 **每个 patch、每个通道内部标准化的像素**。不是原始强度，也不是离散视觉 token 分类。
- 此处只做推理，不更新参数。完整验证集可能耗时；下方可明确选择部分评估。


## 1. 配置运行目录

将 `RUN_DIR` 改为训练命令的 `--output` 目录。也可设置环境变量 `CELLVIT_RUN_DIR`。
默认查看 `latest.pt`；比较最佳重建模型时，将 `CHECKPOINT_NAME` 改成 `best_reconstruction.pt`。
日志查看不依赖 checkpoint；如果训练尚未保存模型，先只运行前半部分。


In [ ]:
import json
import os
from itertools import islice
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from cellvit.data.streaming_dataset import make_dataloader
from cellvit.evaluate import evaluate_reconstruction
from cellvit.models import MAEConfig, MaskedAutoencoder

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
RUN_DIR = Path(os.environ.get(
    "CELLVIT_RUN_DIR",
    "/NFS2_home/NFS2_home_3/xkj2006/cell_painting/project/runs/mae_batch2_20epochs",
))
CHECKPOINT_NAME = "latest.pt"
SPLIT = "val"                  # 先验证，方案确定后再改成 test。
MAX_BATCHES = None             # None = 完整评估；32 = 仅前 32 个 batch 的快速预览。
NUM_WORKERS = 0                # Notebook 中从 0 开始；服务器可自行增大。
FORCE_FP32 = False             # 默认沿用训练精度；改动后不直接与 AMP 分数混比。
PREVIEW_WELLS = 1
RUN_GRAY_DIAGNOSTICS = False  # 可选；会再次遍历数据。
GRAY_MAX_BATCHES = 32          # None = 完整灰度诊断。

print("Run directory:", RUN_DIR.resolve())


## 2. 读取日志

`train_loss` 是一组梯度累积样本的平均 loss；验证指标覆盖整份验证集。训练曲线会更抖动。

- `step`：成功更新次数。
- `update_attempts`：包含 AMP 跳过在内的更新尝试次数；学习率和周期保存按此推进。
- `amp_overflow=True`：这次未更新模型，GradScaler 降低缩放系数后继续。
- `val_reconstruction_loss` / `val_masked_mse`：同一项验证 MSE，保留旧字段以兼容旧日志。

只完成短程训练、尚未跑完一轮时，没有验证记录是正常的。下面兼容旧日志；实时读取时只忽略最后一条尚未写完整的记录。


In [ ]:
log_path = RUN_DIR / "metrics.jsonl"
records = []
if log_path.exists():
    lines = log_path.read_text(encoding="utf-8").splitlines()
    for index, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            if index != len(lines) - 1:
                raise
            print("忽略正在写入的最后一条记录；稍后重新运行本单元格。")
else:
    print("尚无 metrics.jsonl；已有 checkpoint 时可直接运行第 4 节。")

train_df = pd.DataFrame([r for r in records if "train_loss" in r])
val_df = pd.DataFrame([r for r in records if "val_reconstruction_loss" in r])

if not train_df.empty:
    if "update_attempts" not in train_df:
        train_df["update_attempts"] = train_df["step"]
    else:
        train_df["update_attempts"] = train_df["update_attempts"].fillna(train_df["step"])
    display(train_df.tail(5))
if not val_df.empty:
    columns = [c for c in (
        "epoch", "step", "val_reconstruction_loss",
        "val_zero_baseline_mse", "val_reconstruction_skill", "val_wells",
    ) if c in val_df]
    display(val_df[columns].tail(10))
else:
    print("暂无轮末验证记录；第 4 节可以直接评估当前 checkpoint。")


## 3. 训练曲线与重建评分

主误差是隐藏像素的 **Masked MSE**，越低越好。为了方便解释，另外报告：

$$S=1-\frac{\mathrm{MSE}_{model}}{\mathrm{MSE}_{zero}}$$

`MSE_zero` 在完全相同的隐藏位置上计算，基线始终预测 0。在默认标准化目标空间中，0 是局部标准化均值。

- S = 1：隐藏目标重建完全正确。
- S = 0：与全零基线相同。
- S < 0：比基线差。
- S = 0.4：误差相对基线降低 40%，**不是分类准确率 40%**。
- 当全部目标能量接近零时，S 不可定义，输出 `None`，仍报告 MSE。

S 用整份评估数据累计的误差求比值，不先平均各 batch 的百分比。它是相对重建评分，不是绝对原始像素恢复精度。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
if not train_df.empty:
    axes[0].plot(train_df["step"], train_df["train_loss"], alpha=0.2, label="Train (raw)")
    axes[0].plot(
        train_df["step"], train_df["train_loss"].rolling(20, min_periods=1).mean(),
        label="Train (20-record mean)",
    )
    if "amp_overflow" in train_df:
        skipped = train_df["amp_overflow"].fillna(False).astype(bool)
        axes[0].scatter(
            train_df.loc[skipped, "step"], train_df.loc[skipped, "train_loss"],
            marker="x", color="red", label="Skipped AMP update",
        )
if not val_df.empty:
    axes[0].plot(
        val_df["step"], val_df["val_reconstruction_loss"], "o-", label="Validation",
    )
    if "val_zero_baseline_mse" in val_df:
        axes[0].plot(
            val_df["step"], val_df["val_zero_baseline_mse"], "--", label="Zero baseline",
        )
    if "val_reconstruction_skill" in val_df:
        axes[1].plot(
            val_df["step"],
            pd.to_numeric(val_df["val_reconstruction_skill"], errors="coerce") * 100,
            "o-", label="Validation error reduction",
        )
axes[0].set(xlabel="Successful optimizer steps", ylabel="Masked MSE", title="Reconstruction loss")
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set(xlabel="Successful optimizer steps", ylabel="Error reduction vs zero (%)",
            title="Reconstruction skill (not accuracy)")
for ax in axes:
    ax.grid(alpha=0.3)
    if ax.get_legend_handles_labels()[0]:
        ax.legend()
plt.tight_layout()
plt.show()

if not train_df.empty and "loss_scale" in train_df:
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(train_df["update_attempts"], train_df["loss_scale"])
    ax.set(xlabel="Update attempts", ylabel="Loss scale", yscale="log",
           title="GradScaler: occasional decreases can be normal")
    ax.grid(alpha=0.3)
    plt.show()


## 4. Checkpoint、模型结构与官方评估

即使没有完成一个 epoch，也可以加载已保存的 `latest.pt`。本节先从 checkpoint 恢复真实结构，再查看模型摘要并运行重建评估。

评估默认使用训练时的 batch size、seed + 10000 和精度，与轮末验证一致。对比不同模型时，还应保持样本顺序、mask_ratio、目标标准化方式一致。改变 batch size 会改变固定遮挡的分配。

只加载自己训练得到的可信 checkpoint。MDS 清单必须与 checkpoint 一致。CPU 可运行，但完整评估会很慢。


In [ ]:
import hashlib

checkpoint_path = RUN_DIR / CHECKPOINT_NAME
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"尚无 checkpoint：{checkpoint_path}")

print("Loading checkpoint:", checkpoint_path)
state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
if state.get("format") != "cellvit-spatial-mae-v1":
    raise ValueError("不是本项目的 MAE checkpoint")

MDS_ROOT = Path(os.environ.get("RXRX3_MDS_ROOT", state["run_config"]["mds_root"]))
manifest_bytes = (MDS_ROOT / "manifest.json").read_bytes()
manifest = json.loads(manifest_bytes)
if manifest.get("format") != "rxrx3-core-well-mds-v1" or not manifest.get("complete"):
    raise ValueError("需要完整转换的 MDS")
if hashlib.sha256(manifest_bytes).hexdigest() != state["manifest_sha256"]:
    raise ValueError("MDS 清单与训练 checkpoint 不一致")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MaskedAutoencoder(MAEConfig(**state["model_config"]))
model.load_state_dict(state["model"])
model.to(device).eval()

BATCH_SIZE = state["run_config"]["batch_size"]
DATA_SEED = state["run_config"]["seed"]
MASK_SEED = DATA_SEED + 10_000
USE_AMP = state["run_config"]["amp"] and not FORCE_FP32 and device.type == "cuda"

def new_eval_loader():
    # 每次评估/预览重新创建，避免沿用一个已消耗的 Streaming 迭代器。
    return make_dataloader(
        mds_root=MDS_ROOT, split=SPLIT, batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS, shuffle_seed=DATA_SEED,
    )

print({
    "checkpoint": str(checkpoint_path), "step": state["global_step"],
    "device": str(device), "batch_size": BATCH_SIZE,
    "precision": "amp_fp16" if USE_AMP else "fp32",
    "split": SPLIT, "max_batches": MAX_BATCHES,
})
# 释放不用于推理的优化器状态，保留已加载的 model。
del state


### 4.1 模型配置与 torchinfo.summary

hidden size 是每个 token 的特征维度；mlp_ratio 决定 Transformer 内部 MLP 的临时扩张维度。注意力头数必须整除对应 hidden size。模型参数从 checkpoint 自动读取，不在 Notebook 中写死。

若未安装 torchinfo，先运行 %pip install -q torchinfo 并重启内核。摘要只做一次 batch=1 前向，不更新参数。


In [ ]:
cfg = model.config
grid_size = cfg.image_size // cfg.patch_size
num_patches = grid_size ** 2
visible_patches = max(1, int(num_patches * (1 - cfg.mask_ratio)))
masked_patches = num_patches - visible_patches

assert cfg.image_size % cfg.patch_size == 0
assert cfg.encoder_dim % cfg.encoder_heads == 0
assert cfg.decoder_dim % cfg.decoder_heads == 0

config_view = {
    **vars(cfg),
    "encoder_head_dim": cfg.encoder_dim // cfg.encoder_heads,
    "decoder_head_dim": cfg.decoder_dim // cfg.decoder_heads,
    "encoder_mlp_dim": int(cfg.encoder_dim * cfg.mlp_ratio),
    "decoder_mlp_dim": int(cfg.decoder_dim * cfg.mlp_ratio),
}
display(pd.Series(config_view, name="value").to_frame())

architecture = pd.DataFrame([
    ("Input", f"[B, {cfg.in_channels}, {cfg.image_size}, {cfg.image_size}]", "六通道图像"),
    ("Patch embedding", f"[B, {num_patches}, {cfg.encoder_dim}]", f"{cfg.patch_size}×{cfg.patch_size} patch"),
    ("Visible tokens", f"[B, {visible_patches}, {cfg.encoder_dim}]", f"mask={cfg.mask_ratio:.0%}"),
    ("Encoder", f"[B, {visible_patches}, {cfg.encoder_dim}]", f"{cfg.encoder_depth} blocks / {cfg.encoder_heads} heads"),
    ("Decoder projection", f"[B, {visible_patches}, {cfg.decoder_dim}]", f"{cfg.encoder_dim}→{cfg.decoder_dim}"),
    ("Restore mask tokens", f"[B, {num_patches}, {cfg.decoder_dim}]", f"补入{masked_patches}个token"),
    ("Decoder", f"[B, {num_patches}, {cfg.decoder_dim}]", f"{cfg.decoder_depth} blocks / {cfg.decoder_heads} heads"),
    ("Prediction", f"[B, {num_patches}, {model.patch_dim}]", f"{cfg.in_channels}×{cfg.patch_size}×{cfg.patch_size}"),
], columns=["stage", "shape", "meaning"])
display(architecture)


In [ ]:
try:
    from torchinfo import summary
except ImportError as exc:
    raise ImportError("请先运行 %pip install -q torchinfo，然后重启内核。") from exc

class MAESummaryWrapper(torch.nn.Module):
    def __init__(self, mae):
        super().__init__()
        self.mae = mae

    def forward(self, images):
        return self.mae(images)["prediction"]

summary_model = MAESummaryWrapper(model).to(device).eval()
example_images = torch.zeros(
    1, cfg.in_channels, cfg.image_size, cfg.image_size,
    dtype=torch.float32, device=device,
)
with torch.inference_mode():
    model_summary = summary(
        summary_model, input_data=example_images, depth=5,
        col_names=("input_size", "output_size", "num_params", "trainable"),
        row_settings=("var_names",), mode="eval", verbose=0,
    )
print(model_summary)
del example_images, summary_model


### 4.2 官方遮挡重建指标

评估只统计隐藏 patch。Masked MSE/RMSE 越低越好；Reconstruction skill 越高越好。不同模型只有在 split、mask ratio、目标空间、batch size、seed 和精度一致时才能直接比较。


In [ ]:
print("Preparing evaluation loader...")
eval_loader = new_eval_loader()
report = evaluate_reconstruction(
    model, eval_loader, device, seed=MASK_SEED,
    amp=USE_AMP, max_batches=MAX_BATCHES,
)
del eval_loader

expected = manifest["counts"][SPLIT]
complete = report["wells"] == expected
if MAX_BATCHES is None and not complete:
    raise RuntimeError(f"评估数量不完整：{report['wells']} / {expected}")

report.update({
    "checkpoint": str(checkpoint_path), "split": SPLIT,
    "expected_wells": expected, "complete_split": complete,
    "batch_size": BATCH_SIZE,
})
display(pd.DataFrame([{
    "Masked MSE ↓": report["masked_mse"],
    "Masked RMSE ↓": report["masked_rmse"],
    "Zero baseline MSE": report["zero_baseline_mse"],
    "Error reduction vs zero (%) ↑": (
        100 * report["reconstruction_skill"]
        if report["reconstruction_skill"] is not None else np.nan
    ),
    "Wells": report["wells"],
    "Complete split": complete,
}]))
print("完整评估" if complete else f"部分评估：只评价了 {report['wells']} / {expected} 个 well")
print("Target space:", report["target_space"], "| Precision:", report["precision"])
print(f"Elapsed: {report['elapsed_seconds']:.1f}s | "
      f"Speed: {report['wells_per_second']:.2f} wells/s | "
      f"Loader wait: {report['loader_wait_seconds']:.1f}s")


## 5. 各通道表现

各通道均在隐藏位置上评价。标准化缓解通道亮度尺度差异，但每个通道的结构和可预测性仍然不同。某个通道的评分持续很低时，应结合下一节图像检查。

所有通道加权汇总的总评分不等于各通道评分的简单平均。比较实验时不要混用这两种汇总方法。


In [ ]:
channel_df = pd.DataFrame({
    "channel": [f"ch{i + 1}" for i in range(model.config.in_channels)],
    "masked_mse": report["per_channel_mse"],
    "zero_baseline_mse": report["per_channel_zero_baseline_mse"],
    "reconstruction_skill": report["per_channel_reconstruction_skill"],
})
display(channel_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(channel_df))
axes[0].bar(x - 0.18, channel_df["masked_mse"], width=0.36, label="Model")
axes[0].bar(x + 0.18, channel_df["zero_baseline_mse"], width=0.36, label="Zero baseline")
axes[0].set(ylabel="Masked MSE", title="Hidden-patch error by channel")
axes[0].legend()
axes[1].bar(x, pd.to_numeric(channel_df["reconstruction_skill"], errors="coerce") * 100)
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set(ylabel="Error reduction vs zero (%)", title="Reconstruction skill by channel")
for ax in axes:
    ax.set_xticks(x, channel_df["channel"])
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 6. 六通道重建图

这里使用评估集第一批、相同的遮挡种子。每行一个通道：

1. 原图：原始输入 `[0,1]`。
2. 可见输入：灰色区域被遮挡，Encoder 看不到。
3. 真实目标：默认是 patch 内标准化像素，与训练 loss 一致。
4. 补全目标：隐藏区域使用预测，可见区域复制真实目标，仅帮助观察上下文。
5. 绝对误差：只显示隐藏区域。

**第 3、4 列使用相同色阶；指标只计算隐藏部分。** 显示时使用分位数色阶提升可读性，指标计算没有裁剪预测。
这里不使用隐藏区域的真实均值/方差把预测恢复成原始亮度，因此不会把真实强度信息注入预测图。


In [ ]:
preview_loader = new_eval_loader()
batch = next(iter(preview_loader))
images = batch["image"].to(device)
with torch.inference_mode():
    generator = torch.Generator(device=device).manual_seed(MASK_SEED)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        output = model(images, generator=generator)
    target_patches = output["target"]
    prediction_patches = output["prediction"].float().reshape_as(target_patches)
    target_images = model.unpatchify(target_patches).cpu().numpy()
    prediction_images = model.unpatchify(prediction_patches).cpu().numpy()
    pixel_mask = model.unpatchify(
        output["mask"][:, :, None, None].expand_as(target_patches)
    ).bool().cpu().numpy()
raw_images = images.cpu().numpy()
well_ids = batch["well_id"]
del preview_loader, images, output, target_patches, prediction_patches

gray = plt.get_cmap("gray").copy()
gray.set_bad("0.5")
heat = plt.get_cmap("magma").copy()
heat.set_bad("0.5")
target_label = "Standardized target" if model.config.norm_pix_loss else "Raw target"

for well_index in range(min(PREVIEW_WELLS, len(raw_images))):
    channels = model.config.in_channels
    fig, axes = plt.subplots(channels, 5, figsize=(17, 3 * channels), squeeze=False)
    for channel in range(channels):
        raw = raw_images[well_index, channel]
        target = target_images[well_index, channel]
        prediction = prediction_images[well_index, channel]
        hidden = pixel_mask[well_index, channel]
        completed = np.where(hidden, prediction, target)
        error = np.abs(prediction - target)
        raw_lo, raw_hi = np.quantile(raw, [0.01, 0.995])
        target_lo, target_hi = np.quantile(target, [0.01, 0.99])
        raw_hi = max(raw_hi, raw_lo + 1e-6)
        target_hi = max(target_hi, target_lo + 1e-6)

        axes[channel, 0].imshow(raw, cmap=gray, vmin=raw_lo, vmax=raw_hi)
        axes[channel, 1].imshow(
            np.ma.masked_where(hidden, raw), cmap=gray, vmin=raw_lo, vmax=raw_hi,
        )
        axes[channel, 2].imshow(target, cmap=gray, vmin=target_lo, vmax=target_hi)
        axes[channel, 3].imshow(completed, cmap=gray, vmin=target_lo, vmax=target_hi)
        heatmap = axes[channel, 4].imshow(
            np.ma.masked_where(~hidden, error), cmap=heat,
            vmin=0, vmax=max(float(np.quantile(error[hidden], 0.99)), 1e-6),
        )
        fig.colorbar(heatmap, ax=axes[channel, 4], fraction=0.046, pad=0.04)
        axes[channel, 0].set_ylabel(f"ch{channel + 1}")
        for ax in axes[channel]:
            ax.set_xticks([])
            ax.set_yticks([])
    for ax, title in zip(axes[0], [
        "Original input", "Visible input", target_label,
        "Completed target", "Absolute error (hidden)",
    ]):
        ax.set_title(title)
    fig.suptitle(f"Well: {well_ids[well_index]} | mask ratio: {model.config.mask_ratio:.0%}", y=1.01)
    plt.tight_layout()
    plt.show()


## 7. 可选：严格灰度匹配与 Nearest-5

该诊断使用真实隐藏 patch 的均值和标准差把预测还原到 [0,1]，再量化到 0–255。Gray Exact 要求整数灰度完全相等；Nearest-5 要求真值落在预测附近5个合法灰度档位内。

它使用了隐藏真值统计量，会高估独立恢复原始强度的能力，也不是 Top-5 分类准确率。默认关闭，避免官方评估后再次完整遍历数据。


In [ ]:
@torch.inference_mode()
def evaluate_gray_diagnostics(max_batches=32, log_every=100):
    if max_batches is not None and max_batches < 1:
        raise ValueError("max_batches 必须为正整数或 None")
    totals = torch.zeros(model.config.in_channels, 3, dtype=torch.float64, device=device)
    wells = batches = 0
    started = perf_counter()
    loader = new_eval_loader()
    selected = loader if max_batches is None else islice(loader, max_batches)

    for batch_index, batch in enumerate(selected):
        images = batch["image"].to(device, non_blocking=True)
        generator = torch.Generator(device=device).manual_seed(MASK_SEED + batch_index)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            output = model(images, generator=generator)
        target = output["target"].float()
        prediction = output["prediction"].float().reshape_as(target)
        hidden = output["mask"].bool()
        standardized_prediction = prediction[hidden]
        raw_target = model.patchify(images.float())[hidden]

        if model.config.norm_pix_loss:
            mean = raw_target.mean(dim=-1, keepdim=True)
            std = (raw_target.var(dim=-1, unbiased=False, keepdim=True) + 1e-6).sqrt()
            raw_prediction = standardized_prediction * std + mean
        else:
            raw_prediction = standardized_prediction

        true_gray = (raw_target * 255).round().clamp(0, 255).long()
        pred_gray = (raw_prediction * 255).round().clamp(0, 255).long()
        lower = (pred_gray - 2).clamp(0, 251)
        count = torch.full(
            (model.config.in_channels,),
            true_gray.shape[0] * true_gray.shape[2],
            dtype=torch.float64, device=device,
        )
        exact = (pred_gray == true_gray).sum(dim=(0, 2), dtype=torch.float64)
        nearest5 = ((true_gray >= lower) & (true_gray <= lower + 4)).sum(dim=(0, 2), dtype=torch.float64)
        totals += torch.stack([count, exact, nearest5], dim=1)
        wells += len(batch["well_id"])
        batches += 1
        if log_every and (batches == 1 or batches % log_every == 0):
            print(f"{batches} batches | {wells} wells")

    values = totals.cpu().numpy()
    combined = np.vstack([values.sum(axis=0), values])
    scopes = ["Overall"] + [f"ch{i + 1}" for i in range(model.config.in_channels)]
    metrics = pd.DataFrame([
        {
            "scope": scope,
            "Gray_Exact_%": 100 * exact / count,
            "Gray_Nearest5_%": 100 * nearest5 / count,
        }
        for scope, (count, exact, nearest5) in zip(scopes, combined)
    ]).set_index("scope")
    info = {
        "wells": wells, "batches": batches,
        "complete_split": wells == manifest["counts"][SPLIT],
        "uses_true_hidden_patch_statistics": bool(model.config.norm_pix_loss),
        "elapsed_seconds": perf_counter() - started,
    }
    return metrics, info

if RUN_GRAY_DIAGNOSTICS:
    gray_metrics_df, gray_info = evaluate_gray_diagnostics(GRAY_MAX_BATCHES)
    display(pd.Series(gray_info, name="value").to_frame())
    display(gray_metrics_df.round(4))
    ax = gray_metrics_df.drop(index="Overall").plot.bar(figsize=(9, 4), rot=0)
    ax.set(title="Conditional gray matching on hidden pixels", xlabel="Channel", ylabel="Match rate (%)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("灰度诊断未运行。需要时将 RUN_GRAY_DIAGNOSTICS=True。")


## 8. 如何得出结论

报告模板：

> 在固定验证集、75% 空间遮挡、固定 seed、相同 batch size 和精度下，checkpoint 的 masked MSE 为 **实际值**，全零基线 MSE 为 **实际值**，相对基线误差降低 **实际百分比**。覆盖 **实际 well 数**；目标空间为 patch/通道标准化像素。

- 不把 `1 - MSE` 称为准确率，也不把 normalized MSE 当成原始像素的 PSNR。
- `best_reconstruction.pt` 是完整验证集 MSE 最低的已评估模型；旧 checkpoint 恢复后从新评估开始比较。
- 如需更稳健的最终数字，可分别用几个固定 seed 完整评估，再报告均值和波动；模型之间使用同一组种子。
- 若 AMP 推理本身出现非有限预测，应停止并定位；`FORCE_FP32=True` 可辅助排查，但需标记精度变化。
- Notebook 的图是样本检查；完整验证集指标才是重建任务的总体评价。
